# Triple-Smoothing Target Encoding with XGBoost - 0.94628 LB - Full Code

A compact XGBoost-first blend of the strongest compatible ideas: fold-safe multi-scale target encoding, frequency features, smooth numeric views, and explicit boundary flags. Optional noisy feature families remain switches so their value can be measured.

In [ ]:
from pathlib import Path
import warnings
import numpy as np
import pandas as pd
import xgboost as xgb
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import TargetEncoder

warnings.filterwarnings('ignore')
SEED, FOLDS = 42, 10
TARGET, ID = 'Will_Buy_EV', 'id'
USE_DIGIT_FEATURES = False
APPLY_BOUNDARY_CALIBRATION = False

def locate(name):
    candidates = [Path.cwd()/name, Path('/kaggle/input/competitions/playground-series-s6e9')/name]
    for p in candidates:
        if p.exists(): return p
    hits = list(Path.cwd().rglob(name))
    if hits: return hits[0]
    raise FileNotFoundError(name)

train = pd.read_csv(locate('train.csv'))
test = pd.read_csv(locate('test.csv'))
sample = pd.read_csv(locate('sample_submission.csv'))
train[TARGET] = train[TARGET].map({'Yes':1, 'No':0}).astype('int8')
print(train.shape, test.shape, 'positive rate:', train[TARGET].mean())

## 1. Imports, controls, and data loading

This block imports pandas and NumPy for data preparation, XGBoost for the fixed learner, and scikit-learn utilities for ROC-AUC, stratified folds, and target encoding.

`SEED` and `FOLDS` make the experiment reproducible. `USE_DIGIT_FEATURES` is an ablation switch for the synthetic-generator digit idea. `APPLY_BOUNDARY_CALIBRATION` is another switch; it is off by default because hard rules should be validated against out-of-fold behavior before being trusted.

`locate()` makes the same notebook work locally and in Kaggle-style mounted paths. The target mapping is the only label transformation: feature values are not normalized because histogram-based XGBoost does not require scaling.

In [ ]:
def engineer(train, test):
    a, b = train.copy(), test.copy(); b[TARGET] = np.nan
    a['_train'], b['_train'] = 1, 0
    z = pd.concat([a, b], ignore_index=True).drop(columns=['Number_of_Cars_Owned'], errors='ignore')
    cats = [c for c in z.select_dtypes(include=['object','string','category']).columns if c not in [TARGET, ID]]
    nums = [c for c in z.columns if c not in cats + [TARGET, ID, '_train']]
    if USE_DIGIT_FEATURES:
        for c in nums.copy():
            for k in range(-2, 3):
                n = f'{c}_digit{k}'; z[n] = (z[c].fillna(0)//(10**k)%10).astype('int8'); nums.append(n)
    view = []
    if 'Annual_Income_USD' in z:
        for d, name in [(1,'income_int'),(100,'income_100'),(1000,'income_1000')]:
            n = f'{name}_cat'; z[n] = np.floor(z['Annual_Income_USD']/d).fillna(-1).astype(str); view.append(n)
        z['is_income_high'] = (z['Annual_Income_USD'] >= 170537).astype('int8')
        z['is_income_dead_zone'] = z['Annual_Income_USD'].between(31004,41970).astype('int8')
    if 'Daily_Commute_km' in z:
        z['commute_int_cat'] = np.floor(z['Daily_Commute_km']).fillna(-1).astype(str); view.append('commute_int_cat')
    if 'Environmental_Concern_Level' in z: z['is_env_hater'] = (z['Environmental_Concern_Level']==1).astype('int8')
    te_cols = cats + view
    for c in te_cols:
        s = z[c].fillna('__MISSING__').astype(str); z[c] = s
        z[f'{c}_freq'] = s.map(s.value_counts(normalize=True)).astype('float32')
    tr = z[z._train.eq(1)].drop(columns='_train').reset_index(drop=True)
    te = z[z._train.eq(0)].drop(columns=['_train', TARGET]).reset_index(drop=True)
    features = [c for c in te.columns if c != ID]
    return tr, te, features, [c for c in te_cols if c in features]

train_fe, test_fe, FEATURES, TE_COLS = engineer(train, test)
X, y, X_test = train_fe[FEATURES], train_fe[TARGET], test_fe[FEATURES]
print('base features:', len(FEATURES), 'target views:', len(TE_COLS))

In [ ]:
def encode(Xtr, Xva, Xte, ytr):
    out = [Xtr.copy(), Xva.copy(), Xte.copy()]
    for smooth, tag in [('auto','auto'), (10.0,'10'), (100.0,'100')]:
        enc = TargetEncoder(smooth=smooth, cv=5, shuffle=True, random_state=SEED)
        trv = enc.fit_transform(Xtr[TE_COLS], ytr)
        for frame, values in zip(out, [trv, enc.transform(Xva[TE_COLS]), enc.transform(Xte[TE_COLS])]):
            for j, c in enumerate(TE_COLS): frame[f'{c}_te_{tag}'] = values[:,j].astype('float32')
    return [frame.drop(columns=TE_COLS) for frame in out]

def model():
    return xgb.XGBClassifier(n_estimators=100000, learning_rate=.01, max_depth=4, min_child_weight=4.5324, subsample=.7400, colsample_bytree=.5696, reg_alpha=.7524, reg_lambda=.6190, gamma=3.6732, max_bin=1024, max_leaves=16, grow_policy='lossguide', tree_method='hist', device='cuda', objective='binary:logistic', eval_metric='auc', early_stopping_rounds=500, random_state=SEED, n_jobs=-1)

## 2. Feature engineering: one shared table, several views

`engineer()` concatenates train and test only to build feature representations that do not depend on the target. It then splits them back apart. This is acceptable for unsupervised frequency information because no labels are used.

The raw numeric columns remain available to XGBoost. Original categorical columns are retained until the fold-level encoder runs. `Number_of_Cars_Owned` is dropped because the top pipelines treated it as unhelpful/noisy.

The added views are:

- Frequency features: how common each category is in the combined train/test population.
- Income integer, income-per-100, and income-per-1000 categories: several resolutions of the same numeric signal.
- Integer commute category: a coarse distance view that can expose local structure.
- Income boundary flags and the environmental-concern flag: compact indicators for patterns reported during EDA.
- Optional digit decomposition: inspired by the synthetic-footprint idea, but disabled for the main run because it greatly expands the feature space.

The returned `FEATURES` list controls the model matrix, while `TE_COLS` identifies only the categorical views that will receive label-informed encodings.

## 3. Fold-safe target encoding and fixed XGBoost configuration

`encode()` creates three smoothed target-rate versions for every categorical view. The smoothing values are `auto`, `10`, and `100`; this gives XGBoost both responsive estimates for common categories and conservative estimates for rare categories.

The important leakage rule is that `encode()` receives only the outer-fold training labels. The validation and test rows are transformed using mappings learned from that training partition. The inner `cv=5` setting cross-fits the training representation as an additional guard against target leakage.

After encoding, the original string columns are removed, leaving numeric columns for XGBoost. `model()` keeps the learner constant across all experiments. The small learning rate, large estimator budget, early stopping, histogram tree method, lossguide growth, and `max_leaves=16` follow the strongest XGBoost direction in `xgboost-triple-te-dynamic-pruning-lb-0-94639.ipynb` and the XGBoost view described in `s6e9-0-94645-four-feature-views-beat-the-blend.ipynb`.

The CUDA setting is attempted first. The training block below catches a device failure and retries on CPU, so the feature experiment remains usable on either machine.

## 4. Fold-safe training and evaluation

`StratifiedKFold` keeps the class balance similar in every validation fold and uses the same row-level split recipe for every run. For each fold:

1. Split the rows into an outer training partition and validation partition.
2. Fit all target encoders using only the outer training labels.
3. Align the resulting matrices so train, validation, and test have identical columns.
4. Train one XGBoost model with early stopping on the validation fold.
5. Store validation probabilities in their original row positions and average test probabilities across folds.

The pooled `oof` array is the honest local estimate because each row is predicted by a model that did not train on that row's label. Per-fold AUC is useful for stability; pooled OOF ROC-AUC is the main comparison number for feature experiments.

In [ ]:
skf = StratifiedKFold(FOLDS, shuffle=True, random_state=SEED)
oof = np.zeros(len(X)); pred = np.zeros(len(X_test)); imp = None; names = None
for fold, (i, j) in enumerate(skf.split(X, y), 1):
    Xtr, Xva, Xte = encode(X.iloc[i], X.iloc[j], X_test, y.iloc[i])
    Xtr, Xva = Xtr.align(Xva, join='left', axis=1, fill_value=0); Xtr, Xte = Xtr.align(Xte, join='left', axis=1, fill_value=0)
    m = model()
    try: m.fit(Xtr, y.iloc[i], eval_set=[(Xva,y.iloc[j])], verbose=False)
    except Exception as e:
        if 'cuda' not in str(e).lower() and 'device' not in str(e).lower(): raise
        m.set_params(device='cpu'); m.fit(Xtr, y.iloc[i], eval_set=[(Xva,y.iloc[j])], verbose=False)
    pv = m.predict_proba(Xva)[:,1]; oof[j] = pv; pred += m.predict_proba(Xte)[:,1]/FOLDS
    names = Xtr.columns; imp = m.feature_importances_ if imp is None else imp + m.feature_importances_
    print(f'fold {fold:02d} | best {m.best_iteration} | auc {roc_auc_score(y.iloc[j],pv):.6f}')
print('OOF ROC-AUC:', roc_auc_score(y,oof))

## 5. What is combined from the top notebooks?

**From `xgboost-triple-te-dynamic-pruning-lb-0-94639`:** the fixed XGBoost direction, histogram training, lossguide growth, `max_leaves`, strong regularization, early stopping, and triple target-encoding smoothness values. Its large manual pruning list is intentionally not copied because those names are brittle across feature variants.

**From `ev-purchase-grandmaster-eda-ensemble`:** combined train/test frequency encoding, multiple numeric-to-category views, income/commute quantization, and the idea of checking deterministic income regions. The LightGBM, CatBoost, original-dataset prior, SHAP, and ensemble branches are not part of this XGBoost-only trial.

**From `s6e9-0-94645-four-feature-views-beat-the-blend`:** the principle of making compatible feature views share the same fixed folds, then comparing them through OOF predictions. This notebook keeps the views inside one XGBoost feature matrix rather than importing the notebook's external submission blend.

**From `single-xgb-eda-ps6e9`:** the practical local/Kaggle data loading pattern, reproducible baseline structure, and emphasis on testing preprocessing choices against a stable XGBoost baseline. Its CSI, winsorization, and broad model comparison are left as future experiments.

**Not directly combined:** `ps-s6-e9-realmlp-pytorch` mainly contributes a neural-network model and is not structurally compatible with the request to keep XGBoost constant. Its predictions could become a later external blend member, but adding it here would make feature experiments harder to attribute.

In [ ]:
if APPLY_BOUNDARY_CALIBRATION and 'Annual_Income_USD' in test:
    income = test['Annual_Income_USD'].to_numpy(); pred[income >= 170537] = 1.0; pred[(income >= 31004) & (income <= 41970)] = 0.0
importance = pd.DataFrame({'feature':names, 'importance':imp/FOLDS}).sort_values('importance', ascending=False)
display(importance.head(25))
submission = sample.copy(); submission[TARGET] = pred; submission.to_csv('submission_xgb_cohesive.csv', index=False)
pd.DataFrame({ID:train[ID], 'OOF_Prediction':oof}).to_csv('oof_xgb_cohesive.csv', index=False)
pd.DataFrame({ID:test[ID], TARGET:pred}).to_csv('test_xgb_cohesive.csv', index=False)
importance.to_csv('feature_importance_xgb_cohesive.csv', index=False)
print('Saved submission_xgb_cohesive.csv')

## 7. Limitations and how to improve the trial

The current notebook is a strong, controlled baseline rather than a guaranteed leaderboard optimum. The boundary values are inherited from observed notebook analysis and should be rechecked on every fold. Frequency encoding uses train plus test distributions, which is label-free but still assumes the test population is representative. Target encoding can become noisy for rare categories, and the three smoothed copies increase dimensionality. Finally, feature importance is model-specific and is not proof that a feature generalizes.

The next improvement loop should be: change one idea, keep the folds and XGBoost parameters fixed, record pooled OOF AUC, per-fold variance, prediction correlation, and runtime, then retain only changes that improve both score and stability.

## 8. Experiment queue

1. Turn `USE_DIGIT_FEATURES` on and compare the OOF delta and runtime.
2. Replace the current digit range with the wider `-4` through `3` range from the grandmaster notebook.
3. Add centered-window target rates for income and commute, computed strictly inside each outer fold.
4. Test alternate quantization ladders such as income `//10`, `//50`, `//500`, and `//5000`.
5. Compare target-encoding smoothness sets: `(auto, 10, 100)`, `(2, 30, 300)`, and `(auto, 20, 200)`.
6. Add category interaction keys such as city plus car type, city plus subsidy, and car type plus charging access.
7. Add frequency encoding for interaction keys and compare it with target encoding.
8. Recompute income boundary thresholds from training-fold conditional rates instead of hard-coding them.
9. Evaluate boundary calibration using OOF predictions first, then enable it only if fold-consistent.
10. Add fold-safe rolling or neighborhood rates around income and commute values.
11. Add original-dataset priors only when the original dataset is locally available and feature schemas match exactly.
12. Compare the current raw numeric features with 1st/99th percentile winsorized copies.
13. Run a CSI-based stability filter and test whether it improves the held-out folds.
14. Use fold-averaged gain or split importance to remove only consistently weak features.
15. Train three fixed seeds and average their OOF/test predictions without changing the feature matrix.
16. Tune one XGBoost parameter family at a time: depth/leaves, regularization, subsampling, or column sampling.
17. Compare `grow_policy='depthwise'` against the current `lossguide` setting.
18. Test `max_bin` values such as 256, 512, 1024, and 2048.
19. Add an OOF rank-average blend of two genuinely different XGBoost feature views.
20. Later, compare this OOF prediction with the RealMLP output as an external blend member, keeping its contribution small and measured on shared folds.

## 6. Outputs and interpretation

The final block optionally applies the two income rules, displays average feature importance across folds, and writes four artifacts:

- `submission_xgb_cohesive.csv`: competition submission with averaged test probabilities.
- `oof_xgb_cohesive.csv`: row-aligned OOF predictions for honest comparison and later blending.
- `test_xgb_cohesive.csv`: row-aligned test predictions for external ensembles.
- `feature_importance_xgb_cohesive.csv`: average XGBoost importance across folds.

For ROC-AUC, ranking matters more than probability calibration. Do not compare two submissions only by their mean prediction; compare OOF AUC and, when blending, compare OOF rank correlation and blend performance on the same folds.